# EVE Arabic Teacher → Student Distillation

Teacher: Qwen3-4B-Instruct-2507 (cloud GPU only). Student: Qwen3-0.6B (the model family used by the Android app). LoRA response distillation, then INT4 LiteRT-LM export. The notebook refuses CPU-only sessions.

Choose **GPU T4**, enable Internet, then run all cells. The 4B teacher is never part of the Android app.

In [ ]:
!pip install -q -U "transformers>=4.55,<5" "peft>=0.17" "accelerate>=1.10" "bitsandbytes>=0.46" "litert-torch>=0.10" "litert-lm>=0.19" safetensors
import torch, os, json
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU T4 and check GPU quota. Do not train EVE on CPU.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import subprocess, sys
REPO='https://github.com/khyralslh44-hub/WALLE-Offline-New.git'
BRANCH='eve-80mb-compact-brain'
ROOT='/kaggle/working/WALLE-Offline-New'
if not os.path.isdir(ROOT):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,ROOT],check=True)
os.chdir(ROOT)
print('Project source:',ROOT)


In [ ]:
!python eve-model/distillation/train_eve_qwen3_distillation.py --stage full --output-dir /kaggle/working/eve-distillation --epochs 3


In [ ]:
from pathlib import Path
out=Path('/kaggle/working/eve-distillation')
for p in sorted(out.rglob('*')):
    if p.is_file(): print(p.relative_to(out), p.stat().st_size, 'bytes')
print((out/'distillation_manifest.json').read_text() if (out/'distillation_manifest.json').is_file() else 'Manifest not created: inspect previous cells.')
